# Live Car Detection with YOLO26m (Notebook-First, End-to-End)

## 1. Title and Project Overview

This notebook replaces the legacy Haarcascade vehicle detector with a real notebook-first YOLO26m vehicle detection workflow.

What this notebook does:
- downloads `pkdarabi/vehicle-detection-image-dataset` from Kaggle in notebook cells
- verifies the dataset structure, labels, and available split formats
- prefers the dataset's native YOLO format when present
- fine-tunes **YOLO26m** with OOM fallback to **YOLO26s**
- evaluates with mAP50, mAP50-95, precision, recall, and per-class AP50
- runs inference on holdout images and the included sample video
- saves `best.pt`, `best.onnx`, `metrics.json`, and `artifact_manifest.json`

## 2. Problem Statement

Detect vehicles in road-scene imagery and video with bounding boxes suitable for near real-time deployment.

- Input: street and traffic images, plus a sample traffic video
- Output: vehicle bounding boxes, class labels, and confidences
- Task family: vehicle detection
- Target objects: the dataset may contain one or multiple vehicle classes depending on the selected export format

## 3. Why the Chosen Method Is Correct

**Task family:** vehicle detection.

- Bounding-box object detection is the correct method for cars and other road vehicles
- The prompt explicitly prefers YOLO26m, which matches the April 2026 CV rules for trainable detection
- This dataset is distributed in object detection formats, including YOLO-ready exports, so YOLO26m is the most direct fit

## 4. Hardware / Environment Check

In [ ]:
import os, platform, random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()} — {getattr(torch.version, 'cuda', 'N/A')}")
print(f"Device  : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 5. Dependency Installation

In [ ]:
import importlib, subprocess, sys

def ensure_package(import_name: str, pip_name: str | None = None) -> None:
    pip_name = pip_name or import_name
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("ultralytics")
ensure_package("cv2", "opencv-python")
ensure_package("matplotlib")
ensure_package("pandas")
ensure_package("pyyaml", "pyyaml")
ensure_package("kagglehub")
ensure_package("albumentations")
print("All dependencies satisfied.")

## 6. Imports and Configuration

In [ ]:
import json, os, random, shutil
from collections import Counter
from pathlib import Path

import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

PROJECT_DIR   = Path(os.path.dirname(os.path.abspath("__file__")))
DATA_ROOT     = PROJECT_DIR.parents[2] / "data"
RUNS_DIR      = PROJECT_DIR.parents[2] / "runs"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"

for d in (DATA_ROOT, RUNS_DIR, ARTIFACTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"DATA_ROOT    : {DATA_ROOT}")
print(f"ARTIFACTS_DIR: {ARTIFACTS_DIR}")

## 7. Dataset Source Explanation

**Dataset:** Kaggle — `pkdarabi/vehicle-detection-image-dataset`

- Source URL: https://www.kaggle.com/datasets/pkdarabi/vehicle-detection-image-dataset
- The dataset is intended for vehicle detection and tracking tasks
- Kaggle’s description states it is distributed in multiple formats, including YOLO and COCO
- The dataset also includes sample videos for real-time style inference evaluation

**Credential requirements:**
- Set `KAGGLE_USERNAME` + `KAGGLE_KEY` env vars or place `kaggle.json` in `~/.kaggle/`
- Raises a clear error if credentials are missing — no synthetic fallback

In [ ]:
import subprocess

KAGGLE_DATASET = "pkdarabi/vehicle-detection-image-dataset"
DATASET_DIR = DATA_ROOT / "vehicle_detection"
DATASET_DIR.mkdir(parents=True, exist_ok=True)

def check_kaggle_credentials() -> None:
    has_env = os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")
    has_file = (Path.home() / ".kaggle" / "kaggle.json").exists()
    if not has_env and not has_file:
        raise RuntimeError(
            "Kaggle credentials not found.
"
            "Set KAGGLE_USERNAME and KAGGLE_KEY env vars, or place kaggle.json in ~/.kaggle/"
        )

def download_dataset() -> Path:
    check_kaggle_credentials()
    try:
        import kagglehub
        return Path(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception:
        pass
    subprocess.check_call([
        "kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
        "-p", str(DATASET_DIR), "--unzip",
    ])
    return DATASET_DIR

print("Downloading vehicle detection dataset from Kaggle...")
raw_root = Path(download_dataset())
print(f"Raw dataset at: {raw_root}")
print("\nTop-level contents:")
for p in sorted(raw_root.iterdir()):
    print(f"  {p.name}{'/' if p.is_dir() else ''}")

## 8. Dataset Format Discovery

In [ ]:
def find_native_yolo_root(raw_root: Path) -> tuple[Path | None, Path | None]:
    data_yamls = list(raw_root.rglob("data.yaml"))
    for data_yaml in data_yamls:
        try:
            payload = yaml.safe_load(data_yaml.read_text(encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(payload, dict):
            continue
        keys = set(payload.keys())
        if {"train", "val"}.issubset(keys):
            return data_yaml.parent, data_yaml
    return None, None

def find_sample_video(raw_root: Path) -> Path | None:
    for suffix in ["*.mp4", "*.avi", "*.mov", "*.mkv"]:
        matches = list(raw_root.rglob(suffix))
        if matches:
            return matches[0]
    return None

NATIVE_YOLO_ROOT, NATIVE_DATA_YAML = find_native_yolo_root(raw_root)
SAMPLE_VIDEO = find_sample_video(raw_root)

print(f"Native YOLO root : {NATIVE_YOLO_ROOT}")
print(f"Native data.yaml : {NATIVE_DATA_YAML}")
print(f"Sample video     : {SAMPLE_VIDEO}")

if NATIVE_YOLO_ROOT is None or NATIVE_DATA_YAML is None:
    raise RuntimeError(
        "This notebook expects the Kaggle export to include a YOLO-ready `data.yaml`. "
        "If Kaggle changes the layout, inspect the downloaded folders and update the discovery logic."
    )

## 9. Dataset Verification

In [ ]:
data_yaml_payload = yaml.safe_load(NATIVE_DATA_YAML.read_text(encoding="utf-8"))

def resolve_rel_path(base_root: Path, value: str) -> Path:
    candidate = Path(value)
    if candidate.is_absolute():
        return candidate
    return (base_root / candidate).resolve()

TRAIN_DIR = resolve_rel_path(NATIVE_YOLO_ROOT, str(data_yaml_payload["train"]))
VAL_DIR = resolve_rel_path(NATIVE_YOLO_ROOT, str(data_yaml_payload["val"]))
TEST_DIR = resolve_rel_path(NATIVE_YOLO_ROOT, str(data_yaml_payload.get("test", data_yaml_payload["val"])))

def infer_label_dir(image_dir: Path) -> Path:
    if image_dir.name == "images":
        return image_dir.parent / "labels"
    if image_dir.parent.name == "images":
        return image_dir.parent.parent / "labels" / image_dir.name
    return image_dir.parent / "labels"

TRAIN_LABEL_DIR = infer_label_dir(TRAIN_DIR)
VAL_LABEL_DIR = infer_label_dir(VAL_DIR)
TEST_LABEL_DIR = infer_label_dir(TEST_DIR)

names_payload = data_yaml_payload.get("names", [])
if isinstance(names_payload, dict):
    class_names = [names_payload[idx] for idx in sorted(names_payload.keys())]
else:
    class_names = list(names_payload)

def count_items(image_dir: Path, label_dir: Path):
    images = list(image_dir.glob("*.jpg")) + list(image_dir.glob("*.png")) + list(image_dir.glob("*.jpeg"))
    labels = list(label_dir.glob("*.txt")) if label_dir.exists() else []
    return len(images), len(labels)

ti, tl = count_items(TRAIN_DIR, TRAIN_LABEL_DIR)
vi, vl = count_items(VAL_DIR, VAL_LABEL_DIR)
sti, stl = count_items(TEST_DIR, TEST_LABEL_DIR)

print(f"Train: {ti:5d} images, {tl:5d} labels")
print(f"Val  : {vi:5d} images, {vl:5d} labels")
print(f"Test : {sti:5d} images, {stl:5d} labels")
print(f"Classes ({len(class_names)}): {class_names}")

assert ti >= 100, f"Too few training images: {ti}"
assert vi >= 20, f"Too few validation images: {vi}"
assert len(class_names) >= 1, "No classes found in data.yaml"
print("Dataset verification passed.")

## 10. Data Integrity Audit

In [ ]:
train_images = list(TRAIN_DIR.glob("*.jpg")) + list(TRAIN_DIR.glob("*.png")) + list(TRAIN_DIR.glob("*.jpeg"))
audit_sample = random.sample(train_images, min(300, len(train_images)))
corrupt = 0
image_sizes = []
instance_counts = Counter()
box_areas = []

for image_path in audit_sample:
    img = cv2.imread(str(image_path))
    if img is None:
        corrupt += 1
        continue
    h, w = img.shape[:2]
    image_sizes.append((h, w))

    label_path = TRAIN_LABEL_DIR / f"{image_path.stem}.txt"
    if not label_path.exists():
        continue
    text = label_path.read_text(encoding="utf-8").strip()
    for line in text.splitlines():
        parts = line.split()
        if len(parts) != 5:
            continue
        cls_id = int(parts[0])
        bw = float(parts[3])
        bh = float(parts[4])
        instance_counts[cls_id] += 1
        box_areas.append(bw * bh)

print(f"Sampled images  : {len(audit_sample)}")
print(f"Corrupt images  : {corrupt}")
if image_sizes:
    heights = [h for h, _ in image_sizes]
    widths = [w for _, w in image_sizes]
    print(f"Width range     : {min(widths)} - {max(widths)}")
    print(f"Height range    : {min(heights)} - {max(heights)}")
if box_areas:
    print(f"BBox area mean  : {np.mean(box_areas):.6f}")
    print(f"BBox area min   : {np.min(box_areas):.6f}")
    print(f"BBox area max   : {np.max(box_areas):.6f}")

eda_df = pd.DataFrame([
    {"class_id": cls_id, "class_name": class_names[cls_id], "instances": count}
    for cls_id, count in instance_counts.most_common()
])
print(eda_df.head(15).to_string(index=False))

## 11. Sample Visualization — Ground Truth Boxes

In [ ]:
color_map = {idx: tuple(int(v) for v in np.random.randint(0, 255, size=3)) for idx in range(len(class_names))}
sample_images = random.sample(train_images, min(6, len(train_images)))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, image_path in zip(axes, sample_images):
    img_bgr = cv2.imread(str(image_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    label_path = TRAIN_LABEL_DIR / f"{image_path.stem}.txt"
    det_count = 0
    if label_path.exists():
        text = label_path.read_text(encoding="utf-8").strip()
        for line in text.splitlines():
            parts = line.split()
            if len(parts) != 5:
                continue
            cls_id = int(parts[0])
            xc, yc, bw, bh = map(float, parts[1:])
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            color = color_map.get(cls_id, (0, 255, 0))
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img_rgb, class_names[cls_id], (x1, max(12, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
            det_count += 1
    ax.imshow(img_rgb)
    ax.set_title(f"{image_path.name[:22]}\n{det_count} object(s)", fontsize=8)
    ax.axis("off")

plt.suptitle("Training samples with ground-truth vehicle boxes", fontsize=12)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "sample_images.png"), dpi=100)
plt.close(fig)
print("Saved sample_images.png")

## 12. Train/Validation/Test Split Strategy

In [ ]:
split_df = pd.DataFrame({
    "split": ["train", "val", "test"],
    "images": [ti, vi, sti],
    "labels": [tl, vl, stl],
})
print(split_df.to_string(index=False))
print("\nThis notebook uses the dataset's native YOLO split rather than inventing a new split.")

## 13. Preprocessing and Augmentation Strategy

In [ ]:
import albumentations as A

aug = A.Compose([
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(p=0.4),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=8, p=0.3),
], bbox_params=A.BboxParams(format="yolo", label_fields=["class_labels"]))

sample_image = random.choice(train_images)
sample_label = TRAIN_LABEL_DIR / f"{sample_image.stem}.txt"
image_rgb = cv2.cvtColor(cv2.imread(str(sample_image)), cv2.COLOR_BGR2RGB)

bboxes, class_labels = [], []
if sample_label.exists():
    text = sample_label.read_text(encoding="utf-8").strip()
    for line in text.splitlines():
        parts = line.split()
        if len(parts) == 5:
            class_labels.append(int(parts[0]))
            bboxes.append([float(v) for v in parts[1:]])

try:
    aug_result = aug(image=image_rgb, bboxes=bboxes, class_labels=class_labels)
    aug_img = aug_result["image"]
except Exception:
    aug_img = image_rgb

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(cv2.resize(image_rgb, (320, 240)))
axes[0].set_title("Original")
axes[0].axis("off")
axes[1].imshow(cv2.resize(aug_img, (320, 240)))
axes[1].set_title("Augmented (preview)")
axes[1].axis("off")
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "augmentation_preview.png"), dpi=100)
plt.close(fig)
print("Saved augmentation_preview.png")

## 14. Baseline Sanity Check with Pretrained YOLO26m

In [ ]:
from ultralytics import YOLO

baseline_model = YOLO("yolo26m.pt")
val_images = sorted(list(VAL_DIR.glob("*.jpg")) + list(VAL_DIR.glob("*.png")) + list(VAL_DIR.glob("*.jpeg")))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, image_path in zip(axes, val_images):
    res = baseline_model.predict(str(image_path), verbose=False)[0]
    n = len(res.boxes) if res.boxes is not None else 0
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title(f"Pretrained detections: {n}", fontsize=8)
    ax.axis("off")
plt.suptitle("Baseline sanity check — pretrained YOLO26m", fontsize=11)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "sanity_inference.png"), dpi=100)
plt.close(fig)
print("Saved sanity_inference.png")

## 15. Main Model Setup

In [ ]:
PREFERRED_MODEL = "yolo26m.pt"
FALLBACK_MODEL = "yolo26s.pt"

IMG_SIZE = 640
EPOCHS = 25
BATCH = 16
WORKERS = 2

TRAIN_PROJECT = RUNS_DIR / "live_car_detection"
TRAIN_NAME = "yolo26m_vehicle"

LOCAL_DATA_YAML = ARTIFACTS_DIR / "data.yaml"

def normalize_path_for_yaml(path_value: str) -> str:
    return path_value.replace('\\', '/')

normalized_payload = dict(data_yaml_payload)
normalized_payload["path"] = normalize_path_for_yaml(str(NATIVE_YOLO_ROOT))
LOCAL_DATA_YAML.write_text(yaml.safe_dump(normalized_payload, sort_keys=False), encoding="utf-8")

print(f"Model        : {PREFERRED_MODEL}")
print(f"Epochs       : {EPOCHS}")
print(f"Batch        : {BATCH}")
print(f"Classes      : {class_names}")
print(f"Training yaml: {LOCAL_DATA_YAML}")

## 16. Training Loop or Fine-Tuning Pipeline

In [ ]:
from ultralytics import YOLO

def run_training(model_name: str, batch_size: int):
    model = YOLO(model_name)
    return model.train(
        data=str(LOCAL_DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMG_SIZE,
        batch=batch_size,
        workers=WORKERS,
        project=str(TRAIN_PROJECT),
        name=TRAIN_NAME,
        exist_ok=True,
        device=DEVICE,
        seed=SEED,
        verbose=True,
    )

active_model_name = PREFERRED_MODEL
active_batch = BATCH
try:
    train_results = run_training(active_model_name, active_batch)
    print(f"Training complete with {active_model_name}.")
except RuntimeError as exc:
    if "out of memory" in str(exc).lower():
        print(f"OOM with {active_model_name}; retrying with {FALLBACK_MODEL}...")
        torch.cuda.empty_cache()
        active_model_name = FALLBACK_MODEL
        active_batch = max(4, active_batch // 2)
        train_results = run_training(active_model_name, active_batch)
        print(f"Training complete with {active_model_name} (OOM fallback).")
    else:
        raise

## 17. Evaluation

In [ ]:
best_path = TRAIN_PROJECT / TRAIN_NAME / "weights" / "best.pt"
if not best_path.exists():
    raise FileNotFoundError(f"best.pt not found at {best_path}")

best_model = YOLO(str(best_path))
val_metrics = best_model.val(
    data=str(LOCAL_DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    device=DEVICE,
    verbose=True,
)

map50 = float(val_metrics.box.map50)
map50_95 = float(val_metrics.box.map)
precision = float(val_metrics.box.mp)
recall = float(val_metrics.box.mr)

print(f"mAP50     : {map50:.4f}")
print(f"mAP50-95  : {map50_95:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

names = val_metrics.names
cls_list = list(names.values()) if isinstance(names, dict) else list(names)
ap50_vals = val_metrics.box.ap50
class_aps = {}
for cls_name, ap in zip(cls_list, ap50_vals):
    class_aps[cls_name] = float(ap)

metrics_out = {
    "map50": map50,
    "map50_95": map50_95,
    "precision": precision,
    "recall": recall,
    "per_class_ap50": class_aps,
}
with open(ARTIFACTS_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_out, f, indent=2)

print(pd.DataFrame([
    {"class_name": name, "ap50": ap}
    for name, ap in sorted(class_aps.items(), key=lambda item: item[1], reverse=True)
]).head(15).to_string(index=False))
print("Saved metrics.json")

## 18. Holdout Image Inference Examples

In [ ]:
test_images = sorted(list(TEST_DIR.glob("*.jpg")) + list(TEST_DIR.glob("*.png")) + list(TEST_DIR.glob("*.jpeg")))[:6]
if not test_images:
    test_images = val_images

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, image_path in zip(axes, test_images):
    res = best_model.predict(str(image_path), conf=0.25, verbose=False)[0]
    n = len(res.boxes) if res.boxes is not None else 0
    ax.imshow(res.plot()[:, :, ::-1])
    ax.set_title(f"{image_path.name[:22]}\n{n} detections", fontsize=7)
    ax.axis("off")
plt.suptitle("Holdout image inference — fine-tuned YOLO26m", fontsize=11)
plt.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "holdout_inference.png"), dpi=100)
plt.close(fig)
print("Saved holdout_inference.png")

## 19. Video Inference Example

In [ ]:
if SAMPLE_VIDEO is None:
    print("No sample video found in the dataset download.")
else:
    cap = cv2.VideoCapture(str(SAMPLE_VIDEO))
    frames = []
    frame_count = 0
    while cap.isOpened() and len(frames) < 6:
        ok, frame = cap.read()
        if not ok:
            break
        frame_count += 1
        if frame_count % 20 == 0:
            frames.append(frame)
    cap.release()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    for ax, frame in zip(axes, frames):
        res = best_model.predict(frame, conf=0.25, verbose=False)[0]
        ax.imshow(res.plot()[:, :, ::-1])
        ax.axis("off")
    plt.suptitle("Sample video inference frames", fontsize=11)
    plt.tight_layout()
    fig.savefig(str(ARTIFACTS_DIR / "video_inference_frames.png"), dpi=100)
    plt.close(fig)
    print("Saved video_inference_frames.png")

## 20. Error Analysis

In [ ]:
run_dir = TRAIN_PROJECT / TRAIN_NAME
for fname in ["results.png", "PR_curve.png", "F1_curve.png", "confusion_matrix.png"]:
    fpath = run_dir / fname
    if fpath.exists():
        img = cv2.cvtColor(cv2.imread(str(fpath)), cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(fname)
        plt.tight_layout()
        fig.savefig(str(ARTIFACTS_DIR / fname), dpi=80)
        plt.close(fig)
        print(f"Saved {fname}")

if class_aps:
    names_sorted, aps_sorted = zip(*sorted(class_aps.items(), key=lambda item: item[1], reverse=True))
    fig, ax = plt.subplots(figsize=(max(8, len(names_sorted) * 0.8), 5))
    ax.bar(range(len(names_sorted)), aps_sorted, color="steelblue")
    ax.set_xticks(range(len(names_sorted)))
    ax.set_xticklabels(names_sorted, rotation=45, ha="right")
    ax.set_ylabel("AP50")
    ax.set_title("Per-class AP50 on validation set")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    fig.savefig(str(ARTIFACTS_DIR / "per_class_ap50.png"), dpi=100)
    plt.close(fig)
    print("Saved per_class_ap50.png")

## 21. Save Model and Artifacts

In [ ]:
saved_pt = ARTIFACTS_DIR / "best.pt"
shutil.copy2(best_path, saved_pt)
print(f"Saved best.pt -> {saved_pt}")

export_model = YOLO(str(saved_pt))
onnx_path = export_model.export(format="onnx", imgsz=IMG_SIZE, opset=12)
saved_onnx = ARTIFACTS_DIR / "best.onnx"
shutil.copy2(onnx_path, saved_onnx)
print(f"Saved best.onnx -> {saved_onnx}")

with open(ARTIFACTS_DIR / "metrics.json", "r", encoding="utf-8") as f:
    metrics_payload = json.load(f)

manifest = {
    "project": "Live Car Detection (YOLO26m)",
    "dataset": "pkdarabi/vehicle-detection-image-dataset",
    "native_yolo_root": str(NATIVE_YOLO_ROOT),
    "sample_video": str(SAMPLE_VIDEO) if SAMPLE_VIDEO else None,
    "model_file": "best.pt",
    "onnx_file": "best.onnx",
    "metrics": metrics_payload,
}
with open(ARTIFACTS_DIR / "artifact_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("Saved artifact_manifest.json")

## 22. Reproducibility Notes

- Seed `42` is fixed for Python, NumPy, and PyTorch
- The notebook uses the dataset's native split rather than inventing a new one
- The training YAML is copied locally with normalized paths to avoid path resolution issues
- OOM fallback switches from YOLO26m to YOLO26s and halves the batch size

## 23. Conclusion and Limitations

This notebook implements a real notebook-first vehicle detection workflow:

- Real Kaggle download with fail-loud credential checks
- Real verification of the dataset's native YOLO structure and labels
- Real YOLO26m fine-tuning, validation, holdout inference, and video-frame inference
- Real artifact export to `best.pt`, `best.onnx`, `metrics.json`, and `artifact_manifest.json`

**Limitations:**
- The exact class set depends on the Kaggle export selected in the dataset package
- If Kaggle changes the internal folder layout, the format discovery cell may need a small update
- This notebook focuses on detection quality and inference examples; it does not benchmark FPS formally

## 24. How to Improve This Project

1. Add formal FPS benchmarking on the sample video for GPU and CPU.
2. Export and benchmark TensorRT or OpenVINO for deployment.
3. Add class-balanced augmentation if the dataset is skewed toward one vehicle type.
4. Evaluate on a second public road-scene dataset for stronger generalization evidence.
5. Add ByteTrack on top of the trained detector if the next goal is vehicle tracking rather than pure detection.